# Level 3 — 불균형 대응 및 고급 Augmentation

**목표**: 다수 클래스의 정확도를 크게 희생하지 않으면서, 소수 클래스 (foggy / snowy / dawn-dusk) 의 성능을 끌어올립니다.

다음 축에서 **최소 2가지 이상** 의 기법을 적용하세요.
- Loss-level: Weighted CE, Focal Loss, LDAM, Class-Balanced Loss
- Sampling-level: class-balanced sampler
- Augmentation-level: RandAugment, Mixup, CutMix

Level 1 / 2 에서 가장 좋았던 백본을 base 로 사용하세요. wandb 를 사용하면 여러 기법의 비교 Run 을 같은 프로젝트에 모아 볼 수 있어 편리합니다.

In [ ]:
import os
import sys

repo_url  = "https://github.com/min0712-cdl/HYU-AUE8088-PA2.git"
repo_name = "HYU-AUE8088-PA2"

if not os.path.exists(f"/content/{repo_name}"):
    !git clone {repo_url}
else:
    !git -C /content/{repo_name} pull

%cd /content/{repo_name}

from google.colab import drive
drive.mount("/content/drive")

CHECKPOINT_DIR = "/content/drive/MyDrive/AUE8088_PA2/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

%load_ext autoreload
%autoreload 2

# 의존성 설치 (이미 설치된 패키지는 빠르게 skip)
!pip install -q -r requirements.txt

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader

from src.utils.seed import set_seed, seed_worker
from src.utils.transforms import train_transform, eval_transform
from src.utils.trainer import MultiTaskTrainer, TrainConfig
from src.utils.wandb_logger import WandbLogger
from src.utils.metrics import collect_predictions, confusion_matrices, per_class_prf, CLASS_NAMES
from src.datasets.bdd_attr import BDDAttrDataset, ATTRIBUTES
from src.datasets.samplers import class_balanced_sampler
from src.losses.imbalanced import FocalLoss, ClassBalancedLoss, LDAMLoss, weighted_cross_entropy
from src.augment.mix import mixup_data, cutmix_data, mixed_loss
from src.models.vit import vit_small_patch16_224

SEED = 42
set_seed(SEED, deterministic=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
import wandb; wandb.login()   # API key 입력

WANDB_PROJECT = "aue8088-pa2"   # 비활성화하려면 None
WANDB_TAGS    = ["level3"]

In [ ]:
DATA_ROOT = "../data/set_a"
BATCH = 64

# --- 데이터셋 자동 다운로드 (Google Drive) ---------------------------------
# ../data/set_a 가 없으면 zip 을 받아 상위 폴더에 압축 해제 → ../data/set_a, ../data/set_b 생성.
import os, sys, zipfile, subprocess

GDRIVE_FILE_ID = "1L7YC70QlO87aIbE5lbtQ94HUINJijBKK"
ZIP_PATH   = "../aue8088_pa2_data.zip"
EXTRACT_TO = ".."   # zip 내부 최상위가 data/ 이므로 상위 폴더에 풀면 ../data/... 가 됨

if not os.path.isdir(DATA_ROOT):
    try:
        import gdown
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        import gdown

    if not os.path.exists(ZIP_PATH):
        print("데이터셋 zip 다운로드 중...")
        gdown.download(id=GDRIVE_FILE_ID, output=ZIP_PATH, quiet=False)

    print("압축 해제 중...")
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(EXTRACT_TO)
    print(f"완료 → {DATA_ROOT}")
else:
    print(f"데이터셋이 이미 존재합니다 → {DATA_ROOT}")
# --------------------------------------------------------------------------

train_ds = BDDAttrDataset(DATA_ROOT, "train", transform=train_transform())
val_ds   = BDDAttrDataset(DATA_ROOT, "val",   transform=eval_transform())

val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=0, pin_memory=True)

In [ ]:
DEIT_SMALL_URL = (
    "https://dl.fbaipublicfiles.com/deit/"
    "deit_small_patch16_224-cd65a155.pth"
)

def build_pretrained_vit():
    model = vit_small_patch16_224()
    checkpoint = torch.hub.load_state_dict_from_url(
        DEIT_SMALL_URL, map_location="cpu", check_hash=True
    )
    pretrained = checkpoint["model"]
    target = model.state_dict()
    remapped = {}

    for key, value in pretrained.items():
        key = key.replace(".mlp.fc1.", ".mlp.0.")
        key = key.replace(".mlp.fc2.", ".mlp.3.")
        if key.startswith("head."):
            continue
        if key in target and target[key].shape == value.shape:
            remapped[key] = value

    missing, unexpected = model.load_state_dict(remapped, strict=False)
    non_head_missing = [key for key in missing if not key.startswith("head.")]
    if non_head_missing or unexpected:
        raise RuntimeError(
            f"Pretrained backbone mismatch: missing={non_head_missing}, "
            f"unexpected={unexpected}"
        )

    print(f"Matched pretrained tensors: {len(remapped)}")
    return model.to(device), len(remapped)

EPOCHS = 25
LR = 5e-4
WEIGHT_DECAY = 5e-2

EXPERIMENTS = [
    {"name": "baseline", "use_sampler": False, "use_mix": False},
    {"name": "sampler-only", "use_sampler": True, "use_mix": False},
    {"name": "mixup-cutmix-only", "use_sampler": False, "use_mix": True},
    {"name": "sampler+mixup-cutmix", "use_sampler": True, "use_mix": True},
]

In [ ]:
# 옵션 C — 학습 루프에 Mixup/CutMix 를 통합하여 적용
# (깨끗한 실험을 위해서는 _train_one_epoch 를 서브클래싱하는 것이 좋으나,
#  아래는 augmented step 의 핵심만 인라인으로 보인 것입니다.)

from tqdm import tqdm

def step_with_mix(model, loss_fns, images, targets):
    """50% 확률로 Mixup, 나머지 50% 확률로 CutMix 적용."""
    if torch.rand(1).item() < 0.5:
        x, ya, yb, lam = mixup_data(images, targets, alpha=0.2)
    else:
        x, ya, yb, lam = cutmix_data(images, targets, alpha=1.0)
    logits = model(x)
    return mixed_loss(loss_fns, logits, ya, yb, lam)

def make_train_loader(use_sampler):
    if use_sampler:
        sampler = class_balanced_sampler(train_ds, attribute="weather")
        return DataLoader(
            train_ds, batch_size=BATCH, sampler=sampler, num_workers=0,
            pin_memory=True,
        )

    generator = torch.Generator(); generator.manual_seed(SEED)
    return DataLoader(
        train_ds, batch_size=BATCH, shuffle=True, num_workers=0,
        generator=generator, pin_memory=True,
    )

def train_experiment(spec):
    set_seed(SEED, deterministic=True)
    train_loader = make_train_loader(spec["use_sampler"])
    model, matched_tensors = build_pretrained_vit()
    loss_fns = {a: nn.CrossEntropyLoss() for a in ATTRIBUTES}
    optim = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=EPOCHS)

    logger = WandbLogger(
        project=WANDB_PROJECT,
        run_name=f"level3-{spec['name']}",
        config={
            "backbone": "vit_s16_pretrained",
            "pretrained_source": DEIT_SMALL_URL,
            "matched_tensors": matched_tensors,
            "sampler": "class_balanced(weather)" if spec["use_sampler"] else "none",
            "augmentation": "mixup+cutmix" if spec["use_mix"] else "standard",
            "loss": "cross_entropy",
            "epochs": EPOCHS, "batch": BATCH, "lr": LR,
            "weight_decay": WEIGHT_DECAY, "seed": SEED,
        },
        tags=WANDB_TAGS + [spec["name"]],
    )
    trainer = MultiTaskTrainer(
        model, optim, sched, loss_fns, device,
        TrainConfig(epochs=EPOCHS), logger=logger,
    )

    history = {"train_loss": [], "val_avg_mf1": [], "val_per_mf1": []}
    best_score = -1.0
    best_epoch = 0
    best_state = None
    checkpoint_path = os.path.join(CHECKPOINT_DIR, f"level3_{spec['name']}.pth")

    # TODO: step_with_mix 와 trainer.evaluate() 를 사용하여 학습 루프를 작성하세요.
    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0
        num_batches = 0
        pbar = tqdm(train_loader, desc=f"{spec['name']} e{epoch + 1}", leave=False)

        for batch in pbar:
            images = batch["image"].to(device, non_blocking=True)
            targets = {a: batch[a].to(device, non_blocking=True) for a in ATTRIBUTES}
            optim.zero_grad(set_to_none=True)

            with torch.amp.autocast(
                device_type=device.type, enabled=trainer.cfg.amp and device.type == "cuda"
            ):
                if spec["use_mix"]:
                    loss = step_with_mix(model, loss_fns, images, targets)
                else:
                    logits = model(images)
                    loss = sum(loss_fns[a](logits[a], targets[a]) for a in ATTRIBUTES)

            trainer.scaler.scale(loss).backward()
            if trainer.cfg.grad_clip is not None:
                trainer.scaler.unscale_(optim)
                nn.utils.clip_grad_norm_(model.parameters(), trainer.cfg.grad_clip)
            trainer.scaler.step(optim)
            trainer.scaler.update()

            running_loss += loss.item()
            num_batches += 1
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})

        train_loss = running_loss / max(num_batches, 1)
        val_metrics = trainer.evaluate(val_loader)
        sched.step()

        history["train_loss"].append(train_loss)
        history["val_avg_mf1"].append(val_metrics["avg_macro_f1"])
        history["val_per_mf1"].append(val_metrics["per_macro_f1"])

        log_payload = {
            "epoch": epoch + 1, "train/loss": train_loss,
            "val/avg_macro_f1": val_metrics["avg_macro_f1"],
            "lr": optim.param_groups[0]["lr"],
        }
        for attr, value in val_metrics["per_macro_f1"].items():
            log_payload[f"val/mf1_{attr}"] = value
        logger.log(log_payload, step=epoch)

        if val_metrics["avg_macro_f1"] > best_score:
            best_score = val_metrics["avg_macro_f1"]
            best_epoch = epoch + 1
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            torch.save({
                "state_dict": best_state, "history": history, "experiment": spec,
                "best_val_avg_mf1": best_score, "best_epoch": best_epoch,
                "backbone": "vit_s16_pretrained",
                "pretrained_source": DEIT_SMALL_URL,
            }, checkpoint_path)

        print(
            f"[{spec['name']} {epoch + 1:02d}/{EPOCHS}] "
            f"loss={train_loss:.4f} val_avg_MF1={val_metrics['avg_macro_f1']:.4f} "
            f"per={val_metrics['per_macro_f1']}"
        )

    model.load_state_dict(best_state)
    val_pred, _, val_tgt, _ = collect_predictions(model, val_loader, device)
    cms = confusion_matrices(val_pred, val_tgt)
    prf = per_class_prf(val_pred, val_tgt)
    for attr in ATTRIBUTES:
        logger.log_confusion_matrix(f"final/cm_{attr}", cms[attr], CLASS_NAMES[attr])
        rows = list(zip(
            prf[attr]["class"], prf[attr]["precision"], prf[attr]["recall"],
            prf[attr]["f1"], prf[attr]["support"],
        ))
        logger.log_table(
            f"final/prf_{attr}", ["class", "P", "R", "F1", "support"],
            [list(row) for row in rows],
        )

    torch.save({
        "state_dict": best_state, "history": history, "experiment": spec,
        "best_val_avg_mf1": best_score, "best_epoch": best_epoch,
        "backbone": "vit_s16_pretrained", "pretrained_source": DEIT_SMALL_URL,
    }, checkpoint_path)
    logger.finish()

    model = model.cpu()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return {
        "name": spec["name"], "best_val_avg_mf1": best_score,
        "best_epoch": best_epoch, "checkpoint": checkpoint_path,
    }

def completed_result(spec):
    checkpoint_path = os.path.join(CHECKPOINT_DIR, f"level3_{spec['name']}.pth")
    if not os.path.exists(checkpoint_path):
        return None

    checkpoint = torch.load(checkpoint_path, map_location="cpu")
    completed_epochs = len(checkpoint.get("history", {}).get("train_loss", []))
    if completed_epochs < EPOCHS:
        print(
            f"Retraining incomplete run {spec['name']} "
            f"({completed_epochs}/{EPOCHS} epochs)."
        )
        return None

    print(f"Skipping completed run {spec['name']} ({completed_epochs}/{EPOCHS} epochs).")
    return {
        "name": spec["name"],
        "best_val_avg_mf1": checkpoint["best_val_avg_mf1"],
        "best_epoch": checkpoint["best_epoch"],
        "checkpoint": checkpoint_path,
    }

results = []
for spec in EXPERIMENTS:
    result = completed_result(spec)
    results.append(result if result is not None else train_experiment(spec))

In [ ]:
# 전체 실험 중 validation Avg-MF1 이 가장 높은 checkpoint를 Level 5 입력으로 지정.
import shutil

best_result = max(results, key=lambda result: result["best_val_avg_mf1"])
best_path = os.path.join(CHECKPOINT_DIR, "level3_best.pth")
shutil.copy2(best_result["checkpoint"], best_path)

print("Level 3 experiment results:")
for result in results:
    print(
        f"  {result['name']}: best Avg-MF1={result['best_val_avg_mf1']:.4f} "
        f"at epoch {result['best_epoch']}"
    )
print(f"Best checkpoint: {best_path} ({best_result['name']})")

## 분석 (필수)

각 기법에 대해 **속성별 per-class F1 표** 를 작성하세요. 다음 항목을 강조해 주세요.
- 소수 클래스 (foggy / snowy / dawn-dusk) 의 적용 전후 성능 차이.
- 다수 클래스의 회귀 (regression) 발생 여부 — 그 trade-off 가 정당한지 논거.
- Sampling 과 Mixup / CutMix 의 상호작용 — 서로 도움이 되는지 충돌하는지.